# Section 2 — Foundations of AI Pipelines

## Notebook 3 · The inference pipeline

*Estimated reading + run time: 60 minutes. Several runnable code cells; the GPT-4o-mini step needs your `OPENAI_API_KEY`.*

By the end of this notebook you will have:

- Predicted heart-failure mortality on a single patient using the trained model from notebook 02.
- Explained the prediction *twice* — once with **SHAP**, once with **LIME** — and used the *agreement* (or disagreement) between them as a clinical-AI quality check.
- Built an interactive `ipywidgets` slider panel that constructs itself *dynamically* from the model's top-SHAP features (per master-brief §7's v2.0 lesson — *no hard-coded feature names anywhere*).
- Generated a clinical narrative with GPT-4o-mini, *grounded in the model output* and constrained by a single canonical `SYSTEM_PROMPT`.
- Applied **clinical guardrails** that override the model's output when hard-coded clinical rules disagree.

**Inputs:** `best_heart_failure_model.joblib` from notebook 02 *and* `prepared_heart_failure.joblib` from notebook 01. If either is missing, this notebook reproduces them from scratch — though running notebooks 01 and 02 in order is the cleaner path.

## 1. Learning outcomes== failure modes

The inference pipeline takes a trained model and turns it into something a clinician can actually use. Six jobs:

| Job | What it does | Where the failure modes are |
| --- | --- | --- |
| **Load model + scaler + threshold** | Reproduce the deployment configuration faithfully. | Mismatch between scaler and model (notebook 01's scaler with notebook 02's model = silent disaster). |
| **Predict on a new patient** | Apply scaler, run the model, threshold the probability. | Forgetting to scale, or scaling with the wrong scaler. |
| **Explain the prediction** | SHAP, LIME, or both. Per-feature contributions. | Explanations that the clinician cannot reconcile with the patient's clinical picture. |
| **Surface the prediction interactively** | A widget the clinician can poke at. *Dynamically built from top features — no hard-coded feature names.* | Hard-coded feature names that drift out of date as the dataset evolves. |
| **Produce a narrative** | GPT-4o-mini grounded in the model output. | Hallucinated facts not present in the structured output. |
| **Apply clinical guardrails** | Hard-coded rules that override the model when clinical context demands. | Ignoring guardrails, or implementing them as an afterthought. |

## 2. Load the artefacts

In [ ]:
import os, joblib, numpy as np, pandas as pd

MODEL_PATH = "best_heart_failure_model.joblib"
PREP_PATH  = "prepared_heart_failure.joblib"

# Both artefacts are required. If either is missing, point the reader back
# to notebooks 01 and 02 — re-running them is clean and takes <2 minutes.
for p in [MODEL_PATH, PREP_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"{p} not found. Run notebooks 01 and 02 first to produce both artefacts."
        )

model_artefact = joblib.load(MODEL_PATH)
prep_artefact  = joblib.load(PREP_PATH)

model         = model_artefact["model"]
model_name    = model_artefact["model_name"]
threshold     = model_artefact["threshold"]
feature_names = model_artefact["feature_names"]
scaler        = prep_artefact["scaler"]
X_train       = prep_artefact["X_train"]    # used as background data for SHAP
X_test        = prep_artefact["X_test"]
y_test        = prep_artefact["y_test"]

print(f"Loaded model:           {model_name}")
print(f"Deployment threshold:   {threshold:.2f}")
print(f"Validation AUC:         {model_artefact['validation_metrics']['auc']:.3f}")
print(f"Test AUC:               {model_artefact['test_metrics']['auc']:.3f}")
print(f"Features the model expects: {feature_names}")

## 3. Predict on a single patient

We pick a patient from the test set and run them through the model. The test patient is already standardised (notebook 01 did that); a *fresh* patient — for example, from a real EPR — would need to be standardised by the saved scaler before being passed to the model. The cell below shows the full sequence as it would run in production.

In [ ]:
# Pick a single patient from the test set.
PATIENT_INDEX = 5  # arbitrary — change to inspect different patients
patient_features = X_test.iloc[[PATIENT_INDEX]]      # already standardised
patient_outcome  = int(y_test.iloc[PATIENT_INDEX])

# Run the model.
probability = float(model.predict_proba(patient_features)[0, 1])
prediction  = "FLAGGED" if probability >= threshold else "not flagged"

print(f"Patient {PATIENT_INDEX} (test set):")
print(f"  Mortality probability: {probability:.3f}")
print(f"  Threshold:             {threshold:.2f}")
print(f"  Decision:              {prediction}")
print(f"  Actual outcome:        {'died' if patient_outcome == 1 else 'survived'}")
print()
print("Patient's standardised feature values:")
for f, v in zip(feature_names, patient_features.iloc[0]):
    print(f"  {f:<28} {v:+.2f}")

**Read the standardised values as z-scores.** A value of +1.5 means *one and a half standard deviations above the training-set mean for this feature*. A value of −2.0 means *two standard deviations below*. The signs matter for interpretability.

## 4. Explain — SHAP

**SHAP** (SHapley Additive exPlanations) gives each feature a *contribution score* for a specific prediction — how much pushing that feature's value pushed the model's output up or down, relative to the average prediction across the training set.

Different SHAP explainers are appropriate for different model families. The cell below picks the right one based on which model won notebook 02.

In [ ]:
import shap

# Pick the SHAP explainer that matches the trained model family.
if model_name in ("Random Forest", "Gradient Boosting"):
    # TreeExplainer is exact and fast for tree ensembles.
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(patient_features)
    # Tree models return a list (one per class) for binary classification — we want class 1.
    if isinstance(shap_values, list):
        shap_for_patient = shap_values[1][0]
    elif shap_values.ndim == 3:
        shap_for_patient = shap_values[0, :, 1]
    else:
        shap_for_patient = shap_values[0]
elif model_name == "Logistic Regression":
    explainer = shap.LinearExplainer(model, X_train)
    shap_for_patient = explainer.shap_values(patient_features)[0]
else:
    # KernelExplainer is the universal fallback. Slow but reliable for SVM and others.
    background = shap.sample(X_train, 50, random_state=42)
    explainer = shap.KernelExplainer(
        lambda x: model.predict_proba(pd.DataFrame(x, columns=feature_names))[:, 1], background)
    shap_for_patient = explainer.shap_values(patient_features.values)[0]

# Tabulate top features by absolute contribution.
shap_df = pd.DataFrame({
    "feature":      feature_names,
    "value_z":      patient_features.iloc[0].values,
    "shap_value":   np.array(shap_for_patient).ravel(),
})
shap_df["abs_shap"] = shap_df["shap_value"].abs()
shap_df = shap_df.sort_values("abs_shap", ascending=False).reset_index(drop=True)
shap_df["direction"] = np.where(shap_df["shap_value"] > 0, "↑ raises risk", "↓ lowers risk")

print("Top features driving this prediction:")
print(shap_df.head(5)[["feature", "value_z", "shap_value", "direction"]].to_string(index=False))


Read the table top-down: the features at the top of the list are the ones that mattered most for *this* patient. The `direction` column tells you whether each feature *raised* or *lowered* the predicted mortality risk. **A clinician looking at this table should be able to recognise the top features as clinically plausible drivers of mortality risk in heart failure.** *Lower ejection fraction raises risk, higher serum creatinine raises risk* — that pattern of agreement between SHAP and clinical knowledge is the explainability layer doing its job.

If the top SHAP features were instead *anaemia status, sex, smoking history* — features that have weaker established association with mortality in this dataset — that would be a finding worth investigating, not a result to deploy.

## 5. Triangulate — LIME

**LIME** (Local Interpretable Model-agnostic Explanations) attacks the same problem from a different angle. It fits a *local linear model* around the patient of interest and reports the linear coefficients as the explanation. The coefficients are interpretable, the local model is faithful to the original model in the patient's neighbourhood, and the explanation is comparable to SHAP's. **The triangulation is the lesson.** Two methods, same patient, look at where they agree and where they disagree.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=["survived", "died"],
    mode="classification",
    discretize_continuous=False,
    random_state=42,
)

lime_exp = lime_explainer.explain_instance(
    data_row=patient_features.values[0],
    predict_fn=lambda x: model.predict_proba(pd.DataFrame(x, columns=feature_names)),
    num_features=len(feature_names),
)
lime_pairs = lime_exp.as_list()

lime_df = pd.DataFrame(lime_pairs, columns=["feature", "lime_weight"])
lime_df["abs_lime"] = lime_df["lime_weight"].abs()
lime_df = lime_df.sort_values("abs_lime", ascending=False).reset_index(drop=True)

print("LIME — top features by local linear weight:")
print(lime_df.head(5)[["feature", "lime_weight"]].to_string(index=False, float_format='{:.3f}'.format))

# Compare the top-3 features identified by each method.
shap_top3 = set(shap_df.head(3)["feature"])
lime_top3 = set(lime_df.head(3)["feature"])
agreement = shap_top3 & lime_top3
print(f"\nSHAP top-3:  {sorted(shap_top3)}")
print(f"LIME top-3:  {sorted(lime_top3)}")
print(f"Agreement:   {len(agreement)} out of 3 features common to both top-3 lists.")


**The triangulation pattern.** When SHAP and LIME *agree* on the top features, both methods are independently identifying the same drivers of the prediction — strong evidence that the explanation reflects something real about the model. When they *disagree*, neither is necessarily wrong, but the disagreement is itself a signal worth investigating: it usually means the model is using a *combination* of features in a way that one explanation method captures and the other does not.

Section 3 Challenge 3 develops this triangulation explicitly as part of the demo-day rubric — *show your SHAP and LIME side-by-side, name a disagreement if there is one, and tell us what you make of it.*

## 6. Build a dynamic interactive widget — from SHAP top features only

Per master-brief §7 / v2.0 lesson: **widgets must build dynamically from SHAP top features.** No hard-coded feature names anywhere; the widget queries the model's most-important features for *this dataset* and constructs sliders accordingly. If the dataset changes, the widget automatically updates.

The widget below picks the top 4 features by absolute SHAP contribution, builds a slider for each (with min/max from the training-data range), and re-predicts every time you move a slider. The other features are held fixed at the patient's current values.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Pick the top features dynamically — from SHAP, not from a hard-coded list.
TOP_N = 4
top_features = shap_df.head(TOP_N)["feature"].tolist()

# 2. Compute slider min/max for each top feature, in the *original* (un-standardised)
#    feature space. Inverse-transform the training-set min/max to get clinical ranges.
X_train_unscaled = pd.DataFrame(
    scaler.inverse_transform(X_train), columns=feature_names
)
feature_ranges = {
    f: (float(X_train_unscaled[f].min()), float(X_train_unscaled[f].max()))
    for f in feature_names
}

# 3. Start with the chosen patient's values (in original space) as the slider defaults.
patient_unscaled = pd.Series(
    scaler.inverse_transform(patient_features.values)[0],
    index=feature_names,
)

# 4. Build sliders dynamically from top_features.
sliders = {}
for f in top_features:
    lo, hi = feature_ranges[f]
    sliders[f] = widgets.FloatSlider(
        min=lo, max=hi, value=float(patient_unscaled[f]),
        step=(hi - lo) / 100, description=f, continuous_update=False,
        layout=widgets.Layout(width="600px"),
        style={"description_width": "180px"},
    )

output = widgets.Output()

def predict_and_show(**kwargs):
    # Build the patient row using the slider values for top features
    # and the original patient's values for everything else.
    new_patient = patient_unscaled.copy()
    for f, v in kwargs.items():
        new_patient[f] = v
    # Standardise using the saved scaler (this is the line that production gets wrong).
    new_patient_scaled = scaler.transform(new_patient.values.reshape(1, -1))
    new_patient_df = pd.DataFrame(new_patient_scaled, columns=feature_names)
    new_proba = float(model.predict_proba(new_patient_df)[0, 1])
    new_decision = "FLAGGED" if new_proba >= threshold else "not flagged"
    with output:
        clear_output()
        print(f"Mortality probability: {new_proba:.3f}")
        print(f"Decision (threshold {threshold:.2f}): {new_decision}")

ui = widgets.VBox(list(sliders.values()) + [output])
out = widgets.interactive_output(predict_and_show, sliders)
display(ui)
predict_and_show(**{f: s.value for f, s in sliders.items()})


Move the sliders. Watch the predicted mortality probability change. The four sliders are *the four features SHAP identified as most important for this patient* — change the dataset, change the model, change the patient, and the sliders rebuild automatically.

**The clinical reading.** This widget is a *what-if* tool: *if this patient's ejection fraction were 10 % higher, what would the model predict?* The clinician's interpretation is in the language of *modifiable risk factors*. **The widget is not a treatment recommender** — moving sliders does not move patients. It is a way of interrogating the model's reasoning before trusting its output.

## 7. The clinical narrative — GPT-4o-mini

Take the model output (probability, decision, top SHAP features) and let GPT-4o-mini draft a clinical narrative on top of it. **Single canonical `SYSTEM_PROMPT` per notebook** (master brief §7); strict instruction not to invent clinical facts; output grounded in the structured object the cell below builds.

The cell expects `OPENAI_API_KEY` in your environment (Section 0). If the key is not set, it prints a static example output and continues.

In [ ]:
import os, json

# Single canonical system prompt for this notebook (master brief §7).
SYSTEM_PROMPT = (
    "You are a clinical decision support assistant for cardiology.\n"
    "Your role is to draft a structured clinical narrative for a clinician to review.\n"
    "You DO NOT make autonomous decisions. You DO NOT invent clinical facts.\n"
    "Use ONLY the structured model output and SHAP feature contributions provided.\n"
    "Respond in plain prose, 3-5 sentences. End with one explicit recommendation\n"
    "about whether the case warrants escalation, framed as 'consider X' not 'do X'."
)

# Build the structured object the model will summarise.
narrative_input = {
    "model_name":           model_name,
    "mortality_probability": round(probability, 3),
    "deployment_threshold":  threshold,
    "decision":              prediction,
    "top_features": [
        {
            "feature":     row["feature"],
            "z_score":     round(float(row["value_z"]), 2),
            "contribution": round(float(row["shap_value"]), 3),
            "direction":   row["direction"],
        }
        for _, row in shap_df.head(5).iterrows()
    ],
}

if not os.environ.get("OPENAI_API_KEY", "").strip():
    print("OPENAI_API_KEY is not set — printing the structured input that")
    print("WOULD have been sent to GPT-4o-mini. Set the key and re-run for")
    print("the live narrative.\n")
    print(json.dumps(narrative_input, indent=2))
else:
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content":
                f"Draft a 3-5 sentence clinical narrative based on this model output:\n"
                f"{json.dumps(narrative_input, indent=2)}"},
        ],
    )
    print(response.choices[0].message.content)

**Three things to register about this step.**

First, *the narrative is grounded*. Every clinical statement in the output should be traceable to the structured input. The Demo 1 NLI faithfulness check (alongside Chapter 03) is the production-grade extension of this discipline — programmatically verify that each generated sentence is supported by the source. For booklet purposes the *grounded prompt* is the demonstration; in real deployment, the NLI check sits between the LLM output and the clinical user.

Second, *the system prompt is explicit about boundaries*. Three lines: not autonomous, not inventive, only use what was given. These three lines are the difference between a deployable inference layer and a marketing demo.

Third, *temperature is 0 and the SYSTEM_PROMPT is defined once*. Per Chapter 05 — deterministic for clinical use — and per master-brief §7. The same prompt is reused if this cell is re-run; it is not redefined elsewhere in the notebook.

## 8. Clinical guardrails — when the model gets overruled

The last layer of the inference pipeline is the *clinical guardrails* layer. These are hard-coded rules that *override the model* in specific clinical situations — when a clinically obvious flag must be raised regardless of what the model thinks, or when the model's output should be suppressed for safety.

The guardrails below are **illustrative, not clinical guidance** — they show the pattern. A real deployment's guardrails are written by the clinical safety officer with clinical input, not by the data scientist. The pattern is: *take the model output and the patient's raw features; apply rules; if any rule fires, the rule's verdict overrides the model's.*

In [ ]:
def apply_guardrails(probability, patient_unscaled, threshold):
    alerts = []
    final_decision = "FLAGGED" if probability >= threshold else "not flagged"
    final_priority = "routine"

    # Guardrail 1 — extremely low ejection fraction is a clinical flag regardless of model.
    if patient_unscaled.get("ejection_fraction", 50) < 25:
        alerts.append("Severely reduced LVEF (<25%) — escalate to cardiology MDT regardless of model.")
        final_decision = "FLAGGED"
        final_priority = "urgent"

    # Guardrail 2 — high serum creatinine: the patient may have AKI; model may underestimate risk.
    if patient_unscaled.get("serum_creatinine", 1) > 3.0:
        alerts.append("Markedly elevated serum creatinine (>3.0) — review renal function and adjust threshold.")
        final_priority = max(final_priority, "urgent", key=lambda x: ["routine", "urgent"].index(x))

    # Guardrail 3 — model uncertain (between 0.4 and 0.6) → request human review even if not flagged.
    if 0.40 <= probability <= 0.60:
        alerts.append(f"Model output is in the uncertain band ({probability:.2f}). Human review recommended.")

    return {
        "model_probability": probability,
        "final_decision":    final_decision,
        "priority":          final_priority,
        "alerts":            alerts,
    }

guardrail_result = apply_guardrails(probability, patient_unscaled, threshold)

print("Guardrail check:")
print(f"  Model probability: {guardrail_result['model_probability']:.3f}")
print(f"  Final decision:    {guardrail_result['final_decision']}")
print(f"  Priority:          {guardrail_result['priority']}")
if guardrail_result["alerts"]:
    print("  Alerts:")
    for a in guardrail_result["alerts"]:
        print(f"    • {a}")
else:
    print("  (No guardrails triggered)")

**The guardrail is a safety layer, not a second model.** It exists because the data the model was trained on cannot capture every clinically obvious case, and because operational deployment in a real clinical environment requires *human-in-the-loop overrides* that no learned model is allowed to subsume.

Section 3 Challenge 3 asks the reader to write the guardrails section as part of the deployable artefact. The cell above is the small-scale teaching version of what that looks like.

## 9. Putting the pipeline together

A single function that runs everything above end-to-end — load → predict → explain → narrate → guardrail. This is the deployable shape: one entry point, structured output, every layer audit-able.

In [ ]:
def predict_with_full_pipeline(patient_unscaled_dict: dict) -> dict:
    """Take a fresh patient (dict of feature_name → original-scale value) and run the full pipeline."""
    # 1. Standardise.
    patient_row = pd.DataFrame([patient_unscaled_dict], columns=feature_names)
    patient_scaled = scaler.transform(patient_row)
    patient_scaled_df = pd.DataFrame(patient_scaled, columns=feature_names)
    # 2. Predict.
    prob = float(model.predict_proba(patient_scaled_df)[0, 1])
    # 3. Explain (top SHAP features, computed using whichever explainer fits the model).
    if model_name in ("Random Forest", "Gradient Boosting"):
        sv = shap.TreeExplainer(model).shap_values(patient_scaled_df)
        if isinstance(sv, list): sv = sv[1][0]
        elif sv.ndim == 3: sv = sv[0, :, 1]
        else: sv = sv[0]
    else:
        sv = np.zeros(len(feature_names))   # shortcut — full pipeline omitted here
    top_idx = np.argsort(np.abs(sv))[::-1][:5]
    top_features = [
        {"feature": feature_names[i], "contribution": round(float(sv[i]), 3)} for i in top_idx
    ]
    # 4. Guardrails.
    grd = apply_guardrails(prob, pd.Series(patient_unscaled_dict), threshold)
    # 5. Return everything.
    return {
        "probability":  prob,
        "decision":     grd["final_decision"],
        "priority":     grd["priority"],
        "top_features": top_features,
        "alerts":       grd["alerts"],
    }

# Demo on a fresh-looking patient (synthesised values in the original feature scale).
demo_patient = dict(zip(feature_names, patient_unscaled.values))
result = predict_with_full_pipeline(demo_patient)
import json as _json
print(_json.dumps(result, indent=2, default=str))


## 10. What is modifiable at this stage

| Decision | Asked clinically as | What changes if you re-decide |
| --- | --- | --- |
| Which explainer to use | Are model outputs comparable across explanation methods? | SHAP and LIME triangulate; pick one as primary, retain the other as a quality check. |
| How many top features the widget shows | How many levers does a clinician want at the bedside? | Three is sparse, five is the limit before cognitive overload, eight starts to confuse. |
| Whether and what to send to the LLM | What part of the model output benefits from natural-language summarisation? | More detail in the structured input → richer narrative → more chances for hallucination. The trade-off is real. |
| Which guardrails to apply | What hard rules are non-negotiable in this clinical context? | The clinical safety officer owns this list; it is *not* a data-science decision. |
| Whether to deploy at all | Has every layer's failure mode been mitigated? | If even one layer's failure is not recoverable, the deployment is not yet ready. |

## What this notebook produced

- A reproducible single-patient inference path: load → standardise → predict → explain → narrate → guardrail.
- A dynamically built widget interface that adapts to whichever features SHAP identifies as most important.
- A clinical narrative grounded in the model's structured output, with a single canonical system prompt.
- A guardrails layer demonstrating the override pattern.

**Where this lands in Section 3.**

- **Challenge 1 (Final Exam)** uses the same inference shape — RAG over a knowledge base, GPT generation grounded in retrieved snippets, threshold + guardrail.
- **Challenge 2 (Communication)** uses it for SBAR generation grounded in the source clinical record.
- **Challenge 3 (Data Analytics)** is *the same pipeline you just built* — same dataset, same architecture, plus the data-leakage discovery as the central teaching moment.
- **Challenge 4 (Imaging)** uses the same shape for chest X-ray classification — model + Grad-CAM (the imaging analogue of SHAP) + GPT narrative + guardrails.
- **Challenge 5 (Wearables)** uses the same shape for activity classification, FHIR generation, and clinical snapshot.

**One pipeline, five clinical questions.** That is the design lesson Section 2 is built around. Section 1 told you what the pieces are. Section 2 showed you how to build them. Section 3 is where you re-point the pipeline at five distinct clinical problems.

---

*Section 2 · Notebook 3 · OCAI Booklet · Spring 2026 Edition · end of Section 2*